In [3]:

# Local Jupyter setup: run once if the environment is missing packages.
%pip install -q -U datasets transformers accelerate sentence-transformers scikit-learn matplotlib peft trl

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -q --upgrade transformers trl peft accelerate

Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch

# Check if CUDA is available
print(f"CUDA Available: {torch.cuda.is_available()}")

# Get the name of your Nvidia GPU
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch CUDA Version: {torch.version.cuda}")

CUDA Available: True
GPU Device: NVIDIA GeForce RTX 4060 Laptop GPU
PyTorch CUDA Version: 12.4


In [ ]:

import os
import re
import gc
import json
import math
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("m146_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)


In [ ]:
def clear_memory(*names):
    """Delete large variables by name and clear Python/CUDA memory."""
    for name in names:
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


> **Memory tip:** Before loading the GSM8K math model, free GPU memory by running `clear_memory("instruct_model", "instruct_tokenizer")` (or whatever names you used).

## Part B: GSM8K Reasoning

The prompt below is intentionally minimal. You can add your own instructions above the final-answer rule.


In [ ]:
# You may add creative instructions above the final-answer rule.
# If you do nothing, this minimal prompt still runs.
# Keep the final line parseable, or the evaluator may mark correct reasoning as wrong. 
# Do not remove the boxed answer rule. Please edit the first line of the prompt and add your instructions (however long you want).
SYSTEM_PROMPT_FIRST_TRY = """
Your task is to solve the given arithmetic word problem, and return the correct answer alongside reasoning.
For each arithmetic step taken, be sure to explain the operation done, and why.
These intermediate steps should be taken that way it's easier to check work afterwards, and for improved readability.
Number each step, and use latex for clear presentation of each step.
After doing each step, check your work by reversing the calculation to ensure you get the base value.
Do not show us how you checked your work, i.e. do not include the reversed calculations in your final response. Just use them to verify that your answer is correct before returning it.

Final answer rule:
The last line of your response must be exactly:
\\boxed{ANSWER}
""".strip()

SYSTEM_PROMPT = """
Task: Solve the arithmetic word problem below.
Requirements:
 - Numbered Steps: Clearly list each step using LaTeX for all math.
 - Explanations: Explain the operation and the logic behind each step.
 - Accuracy: Internally verify each calculation using inverse operations (do not show this verification in the final output).

 Final answer rule:
The last line of your response must be exactly:
\\boxed{ANSWER}
""".strip()

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig

MATH_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
N_EVAL = 100

gsm8k = load_dataset("openai/gsm8k", "main")
gsm8k_test_100 = gsm8k["test"].select(range(N_EVAL))

def extract_boxed(text):
    idx = text.rfind("\\boxed{")
    if idx == -1:
        return None
    start = idx + len("\\boxed{")
    depth = 1
    pos = start
    while pos < len(text) and depth > 0:
        if text[pos] == "{":
            depth += 1
        elif text[pos] == "}":
            depth -= 1
        pos += 1
    if depth != 0:
        return None
    return text[start:pos - 1]


def extract_gt(raw_answer):
    match = re.search(r"####\s*(.+)", raw_answer)
    return match.group(1).strip().replace(",", "") if match else raw_answer.strip()


def extract_model_answer(text):
    boxed = extract_boxed(text)
    if boxed is not None:
        return boxed.strip()
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else ""


def normalize_answer(ans):
    s = str(ans).strip().replace(",", "").replace("$", "").replace("%", "").replace(" ", "")
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\(left|right|displaystyle)", "", s)
    s = re.sub(r"\\d?frac\{([^}]+)\}\{([^}]+)\}", r"\1/\2", s)
    try:
        v = float(s)
        if math.isfinite(v):
            return str(int(v)) if v == int(v) else str(v)
    except (ValueError, OverflowError):
        pass
    if "/" in s and s.count("/") == 1:
        try:
            a, b = s.split("/")
            return f"{float(a) / float(b):.10g}"
        except (ValueError, ZeroDivisionError):
            pass
    return s


def answers_match(pred, gt):
    p, g = normalize_answer(pred), normalize_answer(gt)
    if p == g:
        return True
    try:
        return abs(float(p) - float(g)) < 1e-6
    except (ValueError, TypeError):
        return False


def load_math_model(model_name=MATH_MODEL, lora_path=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto",
        attn_implementation="eager",
    )
    if lora_path:
        model = PeftModel.from_pretrained(model, lora_path)
    model.eval()
    return model, tokenizer


def make_gsm_prompts(tokenizer, questions, system_prompt=SYSTEM_PROMPT, few_shot_messages=None):
    prompts = []
    for q in questions:
        messages = [{"role": "system", "content": system_prompt}]
        if few_shot_messages:
            messages.extend(few_shot_messages)
        messages.append({"role": "user", "content": q})
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    return prompts


def generate_math(model, tokenizer, questions, system_prompt=SYSTEM_PROMPT, few_shot_messages=None, batch_size=16, max_new_tokens=512):
    responses = []
    for start in range(0, len(questions), batch_size):
        batch_questions = questions[start:start + batch_size]
        prompts = make_gsm_prompts(tokenizer, batch_questions, system_prompt=system_prompt, few_shot_messages=few_shot_messages)
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
        for i in range(len(batch_questions)):
            responses.append(tokenizer.decode(out[i][prompt_len:], skip_special_tokens=True).strip())
    return responses


def evaluate_gsm8k(model, tokenizer, split, num_samples=100, system_prompt=SYSTEM_PROMPT, few_shot_messages=None, batch_size=16, output_name=None):
    n = min(num_samples, len(split))
    records = []
    correct = 0
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        questions = [split[i]["question"] for i in range(start, end)]
        gts = [extract_gt(split[i]["answer"]) for i in range(start, end)]
        responses = generate_math(model, tokenizer, questions, system_prompt=system_prompt, few_shot_messages=few_shot_messages, batch_size=batch_size)
        for offset, (q, gt, resp) in enumerate(zip(questions, gts, responses)):
            pred = extract_model_answer(resp)
            ok = answers_match(pred, gt)
            correct += int(ok)
            records.append({
                "index": start + offset,
                "question": q,
                "ground_truth": gt,
                "model_response": resp,
                "extracted_answer": pred,
                "correct": ok,
            })
    result = {"accuracy": correct / n, "correct": correct, "total": n, "records": records}
    if output_name:
        path = OUTPUT_DIR / output_name
        path.write_text(json.dumps(result, indent=2))
    return result


### B.1-B.2 Base Model Evaluation

For B.1, load the base Qwen2.5-1.5B-Instruct model and evaluate it on the fixed 100 GSM8K test questions.

In [ ]:
# Part B.1

# B.1 is zero-shot evaluation of the base model, not training.
# The project recommends batch size 16, so pass it explicitly here.
math_model, math_tokenizer = load_math_model()
gsm_results = evaluate_gsm8k(
    math_model,
    math_tokenizer,
    gsm8k_test_100,
    system_prompt=SYSTEM_PROMPT,
    batch_size=16,
    output_name="gsm_zero_shot.json",
)

In [ ]:
print(f"Zero-shot GSM8K accuracy result: {gsm_results['accuracy']:.4f}")

In [ ]:
# Part B.2

# find 3 examples where the base model produces the wrong answer
# include the question, excerpt of the model's solution, and the extracted answer vs. ground-truth
incorrect_examples = [r for r in gsm_results["records"] if not r["correct"]][:3]
for i, ex in enumerate(incorrect_examples, 1):
    print(f"Example {i}: --------------------------------------------------------------")
    print(f"Question: {ex['question']}")
    print(f"Model's solution excerpt: {ex['model_response']}")
    print(f"Extracted answer: {ex['extracted_answer']}")
    print(f"Ground truth answer: {ex['ground_truth']}\n")

(Part B.2 continued)

In Example 1, the error occurs from a misunderstanding of the problem, as the solution's first step is to convert to total eggs per week. This is incorrect because all quantities in the problem are per day.

In Example 2, the error occurs from a logical/reasoning error where the model mistakenly thought that the total cost after repairs ($130,000) needed to be multiplied by 2.5 rather than the original cost of the house that it was bought for ($80,000).

In Example 3, another problem misunderstanding occurs; the model thinks that the total number of sprints per week is 3 sprints, when it is actually said in the problem that these 3 sprints are run "3 times a week."

The pattern in these errors seems to be that the model is making logical/reasoning/misunderstanding errors, which all fall under a similar umbrella. It makes sense that the model is not making formatting or arithmetic errors, as these have less "gray area" characteristics than language errors.

In [ ]:

LORA_DEFAULTS = {
    "rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "learning_rate": 2e-4,
    "epochs": 1,
    "batch_size": 8,
    "grad_accum": 4,
    "max_seq_len": 1024,
}


def build_lora_config():
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_DEFAULTS["rank"],
        lora_alpha=LORA_DEFAULTS["lora_alpha"],
        lora_dropout=LORA_DEFAULTS["lora_dropout"],
        target_modules=LORA_DEFAULTS["target_modules"],
        bias="none",
    )


def format_gsm8k_train_example(example):
    reasoning = example["answer"].split("####")[0].strip()
    final = extract_gt(example["answer"])
    assistant = f"{reasoning}\n\n\\boxed{{{final}}}"
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": assistant},
        ]
    }


def train_lora_sft(num_samples, output_dir, model_name=MATH_MODEL):
    output_dir = Path(output_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto",
        attn_implementation="eager",
    )
    model = get_peft_model(model, build_lora_config())
    model.print_trainable_parameters()

    train_data = gsm8k["train"].select(range(min(num_samples, len(gsm8k["train"]))))
    train_data = train_data.map(format_gsm8k_train_example, remove_columns=train_data.column_names)

    args = SFTConfig(
        output_dir=str(output_dir),
        num_train_epochs=LORA_DEFAULTS["epochs"],
        per_device_train_batch_size=LORA_DEFAULTS["batch_size"],
        gradient_accumulation_steps=LORA_DEFAULTS["grad_accum"],
        learning_rate=LORA_DEFAULTS["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        bf16=torch.cuda.is_available(),
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        max_length=LORA_DEFAULTS["max_seq_len"],
        completion_only_loss=True,
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=train_data, processing_class=tokenizer)
    trainer.train()

    adapter_path = output_dir / "final_adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    return str(adapter_path)


### B.3-B.7 LoRA Fine-Tuning

The training utilities are provided below. Run the 1k and 2k experiments yourself and record the results in your report.

> **Memory tip:** Before LoRA training (and between LoRA runs), free GPU memory by running `clear_memory("math_model", "math_tokenizer")` so the base model isn't still resident.

For B.5, train the provided LoRA template on 1,000 GSM8K training examples, then evaluate on the same 100-question test subset.

Write your scaling hypothesis before running 2k. Do you expect a meaningful improvement over 1k? Why?


For B.7, train the provided LoRA template on 2,000 examples and evaluate on the same 100-question test subset.

Make the B.7 accuracy-vs-training-size plot yourself using your results for 0, 1000, and 2000 training examples.


### B.10-B.11 Few-Shot Prompting


In [ ]:

def few_shot_messages_from_train(train_split, indices=(0, 1, 2)):
    messages = []
    for idx in indices:
        ex = train_split[int(idx)]
        reasoning = ex["answer"].split("####")[0].strip()
        final = extract_gt(ex["answer"])
        messages.append({"role": "user", "content": ex["question"]})
        messages.append({"role": "assistant", "content": f"{reasoning}\n\n\\boxed{{{final}}}"})
    return messages

For B.10, build one fixed set of 3 GSM8K training examples and use it for both the base model and the 2k LoRA model. Then compare zero-shot and 3-shot accuracy.

### Bonus: Prompt Playground

For the bonus, start from the original prompt, then write your own prompt variants. You do not need to stop at three trials if you want to keep experimenting.

In [ ]:
# Keep PROMPT_0 as the old prompt. Fill in PROMPT_1, PROMPT_2, PROMPT_3 yourself.
PROMPT_0 = SYSTEM_PROMPT

PROMPT_1 = """
""".strip()

PROMPT_2 = """
""".strip()

PROMPT_3 = """
""".strip()


def evaluate_one_prompt(model, tokenizer, prompt, name="prompt", n=50):
    result = evaluate_gsm8k(
        model,
        tokenizer,
        gsm8k["test"].select(range(n)),
        num_samples=n,
        system_prompt=prompt,
        batch_size=16,
    )
    return {"prompt_name": name, "accuracy": result["accuracy"], "correct": result["correct"], "total": result["total"]}, result

# Baseline bonus run using the old prompt. After this, run your own prompt variants.
if "math_model" not in globals() or "math_tokenizer" not in globals():
    math_model, math_tokenizer = load_math_model()

baseline_row, baseline_result = evaluate_one_prompt(math_model, math_tokenizer, PROMPT_0, name="old_prompt")
baseline_row
